In [ ]:
%pip install "crewai==0.28.8" "crewai_tools==0.1.6"
%pip install "langchain-openai>=0.1.0"
%pip install "langchain-community>=0.2.0"
%pip install "langchain-groq>=0.1.0"
%pip install sentence-transformers
%pip install tavily-python

In [ ]:
%pip install langchain-huggingface

In [28]:
#%pip install langchain langchain-chroma pypdf

In [1]:
from langchain_openai import ChatOpenAI
from crewai_tools import PDFSearchTool
from langchain_community.tools.tavily_search import TavilySearchResults
from crewai import Crew, Task, Agent
from crewai.tools import tool
import os

In [2]:
GROQ_API_KEY = "your-groq-api-key-here"
TAVILY_API_KEY = "your-tavily-api-key-here"

os.environ["GROQ_API_KEY"] = GROQ_API_KEY
os.environ["OPENAI_API_KEY"] = GROQ_API_KEY
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY

In [3]:
from crewai.llm import LLM

llm = LLM(
    model="groq/llama-3.3-70b-versatile",
    max_tokens=300,
    temperature=0.1,
)

In [4]:
#%pip install langchain-text-splitters

In [5]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from crewai.tools import tool

loader = PyPDFLoader("doc.pdf")
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
vectorstore = Chroma.from_documents(chunks, embeddings)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

@tool
def search_document(question: str) -> str:
    """Search the uploaded PDF document for information relevant to the question."""
    docs = retriever.invoke(question)
    return "\n\n".join([d.page_content for d in docs])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
result = search_document.run("What does Sporo Health do?")
print(result)

storage of LLMs in 2024 such as Google’s Med-Gemini, Meta’s Llama 3, OpenAI’s ChatGPT4, or Anthropic’s Claude
3.5, has allowed for these models to process the enormous portions of information for summarization and analysis.
In recognition of the stringent accuracy, the need for personalization, privacy regulations, and the high knowledge
floor needed for AI in clinical workflow, the innovation space gave birth to companies like Sporo Health to combat

ASSESSING THE ROLE OF CLINICAL SUMMARIZATION AND
PATIENT CHART REVIEW WITHIN COMMUNICATIONS , M EDICAL
MANAGEMENT , AND DIAGNOSTICS
Chanseo Lee
Yale School of Medicine and Sporo Health
New Haven, CT 06510
chanseo.lee@yale.edu
Kimon-Aristotelis V ogt
Sporo Health
Boston, MA 02134
kvogt@sporohealth.com
Sonu Kumar
Sporo Health
Boston, MA 02134
sonu@sporohealth.com
July 25, 2024
ABSTRACT
Effective summarization of unstructured patient data in electronic health records (EHRs) is crucial

summarization tasks, and its transformative impact on th

In [7]:
docs = vectorstore.similarity_search("What does Sporo Health do?", k=3)
for i, doc in enumerate(docs):
    print(f"--- Chunk {i+1} ---")
    print(doc.page_content)
    print()

--- Chunk 1 ---
storage of LLMs in 2024 such as Google’s Med-Gemini, Meta’s Llama 3, OpenAI’s ChatGPT4, or Anthropic’s Claude
3.5, has allowed for these models to process the enormous portions of information for summarization and analysis.
In recognition of the stringent accuracy, the need for personalization, privacy regulations, and the high knowledge
floor needed for AI in clinical workflow, the innovation space gave birth to companies like Sporo Health to combat

--- Chunk 2 ---
ASSESSING THE ROLE OF CLINICAL SUMMARIZATION AND
PATIENT CHART REVIEW WITHIN COMMUNICATIONS , M EDICAL
MANAGEMENT , AND DIAGNOSTICS
Chanseo Lee
Yale School of Medicine and Sporo Health
New Haven, CT 06510
chanseo.lee@yale.edu
Kimon-Aristotelis V ogt
Sporo Health
Boston, MA 02134
kvogt@sporohealth.com
Sonu Kumar
Sporo Health
Boston, MA 02134
sonu@sporohealth.com
July 25, 2024
ABSTRACT
Effective summarization of unstructured patient data in electronic health records (EHRs) is crucial

--- Chunk 3 ---
summariz

In [8]:
#%pip install -U langchain-tavily

In [9]:
from crewai.tools import tool
from langchain_tavily import TavilySearch

tavily_search = TavilySearch(k=3, tavily_api_key=TAVILY_API_KEY)

@tool
def search_web(question: str) -> str:
    """Search the web for current, up-to-date information not found in the document."""
    result = tavily_search.invoke(question)
    return str(result)

In [10]:
@tool
def route_question(question: str) -> str:
    """Routes a question to either vectorstore or web_search."""
    keywords = ["document", "pdf", "text", "paper", "article",
                "author", "according to", "mentioned", "states", "describes"]
    if any(k in question.lower() for k in keywords):
        return 'vectorstore'
    return 'web_search'

In [11]:
#%pip install litellm

In [12]:
Router_Agent = Agent(
    role='Query Router',
    goal='Determine the best source to answer the user question',
    backstory=(
        "You are an expert at analysing questions and deciding whether they "
        "are best answered using the content of an uploaded document or by "
        "searching the web. You always use the router_tool to make your decision. "
        "Default to vectorstore unless the question is clearly about current events."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
    tools=[route_question],
)

In [13]:
Retriever_Agent = Agent(
    role='Information Retriever',
    goal='Retrieve the most relevant information to answer the user question',
    backstory=(
        "You are an expert at retrieving information from documents and the web. "
        "When directed to use the vectorstore, you use rag_tool to search the "
        "uploaded document. When directed to use web search, you use web_search_tool. "
        "You always retrieve information before forming any answer."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
    tools=[search_document, search_web],
)

In [14]:
Grader_agent = Agent(
    role='Relevance Grader',
    goal='Assess whether the retrieved information is relevant to the question',
    backstory=(
        "You are an expert evaluator who checks whether retrieved content "
        "actually addresses the user question. You are strict but fair — "
        "partial answers still count as relevant if they help answer the question."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

In [15]:
hallucination_grader = Agent(
    role='Hallucination Grader',
    goal='Verify that the answer is grounded in retrieved facts',
    backstory=(
        "You are an expert fact-checker who verifies that every claim in an answer "
        "is supported by the retrieved source material. You flag any information "
        "that was not present in the retrieved content as a hallucination."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

In [16]:
answer_grader = Agent(
    role='Answer Synthesiser',
    goal='Produce a final, accurate, and useful answer to the user question',
    backstory=(
        "You are an expert communicator who takes verified information and "
        "produces clear, concise, well-structured answers. If the verified "
        "information is insufficient, you perform a web search to supplement it. "
        "You always end your answer by stating whether it came from the document "
        "or the web."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
    tools=[search_web],
)

In [17]:
router_task = Task(
    description=(
        "Analyse the following question and decide the best source to answer it: {question}\n"
        "Use the route_question tool to make your decision.\n"
        "Return ONLY one of these two words, nothing else:\n"
        "- 'vectorstore' if the question is about the content of the uploaded document\n"
        "- 'websearch' if the question requires current information or general knowledge"
    ),
    expected_output=(
        "A single word — either 'vectorstore' or 'websearch'. "
        "No punctuation, no explanation, no preamble. Just the single word."
    ),
    agent=Router_Agent,
    tools=[route_question],
)

In [18]:
retriever_task = Task(
    description=(
        "Retrieve information to answer the question: {question}\n"
        "First check the router_task output:\n"
        "- If 'vectorstore': call search_document with the question as input\n"
        "- If 'websearch': call search_web with the question as input\n"
        "Always retrieve information before forming any response."
    ),
    expected_output=(
        "A factual summary of the retrieved information relevant to the question. "
        "Include specific details, numbers, and quotes where available. "
        "State clearly whether the information came from the document or the web. "
        "Minimum 3 sentences."
    ),
    agent=Retriever_Agent,
    context=[router_task],
)

In [19]:
grader_task = Task(
    description=(
        "Evaluate the retriever_task output for the question: {question}\n"
        "Decide: does the retrieved content actually help answer the question?\n"
        "Grade as relevant if it directly or partially answers the question."
    ),
    expected_output=(
        "Return ONLY the word 'yes' or 'no'.\n"
        "'yes' means: the retrieved content directly or partially answers the question.\n"
        "'no' means: the retrieved content is off-topic or completely irrelevant.\n"
        "No explanation, no punctuation, just 'yes' or 'no'."
    ),
    agent=Grader_agent,
    context=[retriever_task],
)

In [20]:
hallucination_task = Task(
    description=(
        "Check the retriever_task answer for the question: {question}\n"
        "Verify every factual claim is supported by the retrieved source content.\n"
        "Flag it as hallucinated if the answer contains claims not present in the sources."
    ),
    expected_output=(
        "Return ONLY the word 'yes' or 'no'.\n"
        "'yes' means: all claims in the answer are supported by the retrieved content.\n"
        "'no' means: the answer contains fabricated or unsupported claims.\n"
        "No explanation, no punctuation, just 'yes' or 'no'."
    ),
    agent=hallucination_grader,
    context=[grader_task],
)

In [21]:
answer_task = Task(
    description=(
        "Produce the final answer to: {question}\n"
        "Check the hallucination_task output:\n"
        "- If 'yes': the answer is verified — summarise the retriever_task answer clearly and concisely\n"
        "- If 'no': the answer is unreliable — use search_web to find a better answer\n"
        "- If web search also fails: respond with 'Sorry, I could not find a reliable answer.'"
    ),
    expected_output=(
        "A clear, well-structured answer to the question. "
        "Use bullet points or numbered lists where appropriate. "
        "Include specific facts, numbers, or quotes from the sources. "
        "End with a one-sentence source attribution: "
        "either 'Source: document' or 'Source: web search'."
    ),
    context=[router_task, retriever_task, grader_task, hallucination_task],
    agent=answer_grader,
)

In [22]:
rag_crew = Crew(
    agents=[Router_Agent, Retriever_Agent, Grader_agent,
            hallucination_grader, answer_grader],
    tasks=[router_task, retriever_task, grader_task,
           hallucination_task, answer_task],
    verbose=True,
)

In [23]:
inputs = {"question": "What is this document about?"}
result = rag_crew.kickoff(inputs=inputs)
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: e887c5ff-44ba-46d1-97d3-cba8f5751d55                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyse the following question and decide the best source to answer it: What is this document about?     │
│  Use the route_question tool to make your decision.                                                             │
│  Return ONLY one of these two words, nothing else:                                                              │
│  - 'vectorstore' if the question is about the content of the uploaded document                                  │
│  - 'websearch' if the question requires current information or general knowledge                                │
│  ID: bcf14af5-0425-4095-87e4-ec66a63194b0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Query Router                                                                                            │
│                                                                                                                 │
│  Task: Analyse the following question and decide the best source to answer it: What is this document about?     │
│  Use the route_question tool to make your decision.                                                             │
│  Return ONLY one of these two words, nothing else:                                                              │
│  - 'vectorstore' if the question is about the content of the uploaded document                                  │
│  - 'websearch' if the question requires current information or general knowledge                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool route_question executed with result: vectorstore...


╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: route_question                                                                                           │
│  Args: {'question': 'What is this document about?'}                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Produce the final answer to: What is this document about?                                                │
│  Check the hallucination_task output:                                                                           │
│  - If 'yes': the answer is verified — summarise the retriever_task answer clearly and concisely                 │
│  - If 'no': the answer is unreliable — use search_web to find a better answer                                   │
│  - If web search also fails: respond with 'Sorry, I could not find a reliable answer.'                          │
│  Agent: Answer Synthesiser                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'task_started' (expected 
'crew_kickoff_started')

The document is about the importance of accurate and efficient communication in healthcare, particularly in relation to patient charts and electronic health records (EHRs). According to the text, the information housed within patient charts is critical to avoiding misdiagnoses or contraindicatory management plans. The document also mentions efforts to implement natural language processing of clinical narratives into workflow and diagnostics, including in managing coronary artery disease. Additionally, the text cites various sources, including the Sentinel Event Alert and the Agency for Healthcare Research and Quality, to highlight the risks of inadequate hand-off communication and the importance of improving communication in healthcare. The document also references a study on the time spent by internal medicine residents on electronic health records, which found that they spent a significant amount of time on EHRs. The information in this document comes from the document itself, which 

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: e887c5ff-44ba-46d1-97d3-cba8f5751d55                                                                       │
│  Final Output: The document is about the importance of accurate and efficient communication in healthcare,      │
│  particularly in relation to patient charts and electronic health records (EHRs). According to the text, the    │
│  information housed within patient charts is critical to avoiding misdiagnoses or contraindicatory management   │
│  plans. The document also mentions efforts to implement natural language processing of clinical narratives      │
│  into workflow and diagnostics, including in managing coronary artery disease. Additionally, the text cites     │
│  various sources, including the Sentinel Event Alert and the Agency for Healthcare Research and Quality, to     │
│  highlight the risks of inadequate hand-off communication and the importance of improving communication in      │
│  healthcare. The document also references a study on the time spent by internal medicine residents on           │
│  electronic health records, which found that they spent a significant amount of time on EHRs. The information   │
│  in this document comes from the document itself, which was retrieved using the search_document function. The   │
│  document provides specific details, numbers, and quotes, such as the PMID numbers for various studies and the  │
│  doi number for a journal article. Overall, the document provides a detailed analysis of the importance of      │
│  communication in healthcare and the role of EHRs in improving patient care. The information retrieved from     │
│  the document meets the requirements, providing a factual summary of the document's content, including          │
│  specific details, numbers, and quotes. The document's content is relevant to the question of what the          │
│  document is about, and the information comes from the document itself, which was retrieved using the           │
│  search_document function.                                                                                      │
│  Source: document                                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [24]:
inputs = {"question": "According to the document, what percentage of physicians cite EHR workflow as causing burnout?"}
result = rag_crew.kickoff(inputs=inputs)
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: e887c5ff-44ba-46d1-97d3-cba8f5751d55                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyse the following question and decide the best source to answer it: According to the document, what  │
│  percentage of physicians cite EHR workflow as causing burnout?                                                 │
│  Use the route_question tool to make your decision.                                                             │
│  Return ONLY one of these two words, nothing else:                                                              │
│  - 'vectorstore' if the question is about the content of the uploaded document                                  │
│  - 'websearch' if the question requires current information or general knowledge                                │
│  ID: bcf14af5-0425-4095-87e4-ec66a63194b0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Query Router                                                                                            │
│                                                                                                                 │
│  Task: Analyse the following question and decide the best source to answer it: According to the document, what  │
│  percentage of physicians cite EHR workflow as causing burnout?                                                 │
│  Use the route_question tool to make your decision.                                                             │
│  Return ONLY one of these two words, nothing else:                                                              │
│  - 'vectorstore' if the question is about the content of the uploaded document                                  │
│  - 'websearch' if the question requires current information or general knowledge                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool route_question executed with result: vectorstore...

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: route_question                                                                                           │
│  Args: {'question': 'According to the document, what percentage of physicians cite EHR workflow as causing      │
│  burnout?'}                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Produce the final answer to: According to the document, what percentage of physicians cite EHR workflow  │
│  as causing burnout?                                                                                            │
│  Check the hallucination_task output:                                                                           │
│  - If 'yes': the answer is verified — summarise the retriever_task answer clearly and concisely                 │
│  - If 'no': the answer is unreliable — use search_web to find a better answer                                   │
│  - If web search also fails: respond with 'Sorry, I could not find a reliable answer.'                          │
│  Agent: Answer Synthesiser                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'task_started' (expected 
'crew_kickoff_started')

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: e887c5ff-44ba-46d1-97d3-cba8f5751d55                                                                       │
│  Final Output: According to the document, 75% of physicians facing burnout cite EHR workflow as the main        │
│  contributor. The document states that "75% of physicians facing burnout cite EHR workflow as the main          │
│  contributor," especially in primary care. This information comes from the document itself, which was           │
│  retrieved using the search_document function. The document also provides specific details and numbers, such    │
│  as the fact that physicians spend 49.2% of their time per day with EHRs, while only 27% is dedicated to        │
│  face-to-face time with patients. Additionally, the document mentions that burnout rates are the highest at     │
│  50%. The information retrieved from the document meets the requirements, providing a factual summary of the    │
│  document's content, including specific details, numbers, and quotes. The document's content is relevant to     │
│  the question of what percentage of physicians cite EHR workflow as causing burnout, and the information comes  │
│  from the document itself, which was retrieved using the search_document function. The document also            │
│  references various studies, including one published in the journal Z Evid Fortbild Qual Gesundhwes, which      │
│  discusses the issue of information overload in healthcare. Overall, the document provides a detailed analysis  │
│  of the impact of EHR workflow on physician burnout, and the information retrieved from the document provides   │
│  a clear answer to the question.                                                                                │
│  Source: document                                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

According to the document, 75% of physicians facing burnout cite EHR workflow as the main contributor. The document states that "75% of physicians facing burnout cite EHR workflow as the main contributor," especially in primary care. This information comes from the document itself, which was retrieved using the search_document function. The document also provides specific details and numbers, such as the fact that physicians spend 49.2% of their time per day with EHRs, while only 27% is dedicated to face-to-face time with patients. Additionally, the document mentions that burnout rates are the highest at 50%. The information retrieved from the document meets the requirements, providing a factual summary of the document's content, including specific details, numbers, and quotes. The document's content is relevant to the question of what percentage of physicians cite EHR workflow as causing burnout, and the information comes from the document itself, which was retrieved using the search_

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [25]:
inputs = {"question": "What AI tools exist for clinical documentation?"}
result = rag_crew.kickoff(inputs=inputs)
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: e887c5ff-44ba-46d1-97d3-cba8f5751d55                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyse the following question and decide the best source to answer it: What AI tools exist for          │
│  clinical documentation?                                                                                        │
│  Use the route_question tool to make your decision.                                                             │
│  Return ONLY one of these two words, nothing else:                                                              │
│  - 'vectorstore' if the question is about the content of the uploaded document                                  │
│  - 'websearch' if the question requires current information or general knowledge                                │
│  ID: bcf14af5-0425-4095-87e4-ec66a63194b0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Query Router                                                                                            │
│                                                                                                                 │
│  Task: Analyse the following question and decide the best source to answer it: What AI tools exist for          │
│  clinical documentation?                                                                                        │
│  Use the route_question tool to make your decision.                                                             │
│  Return ONLY one of these two words, nothing else:                                                              │
│  - 'vectorstore' if the question is about the content of the uploaded document                                  │
│  - 'websearch' if the question requires current information or general knowledge                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool route_question executed with result: vectorstore...

╭──────────────────────────────────────── 🔧 Tool Execution Started (#3) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: route_question                                                                                           │
│  Args: {'question': 'What AI tools exist for clinical documentation?'}                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Produce the final answer to: What AI tools exist for clinical documentation?                             │
│  Check the hallucination_task output:                                                                           │
│  - If 'yes': the answer is verified — summarise the retriever_task answer clearly and concisely                 │
│  - If 'no': the answer is unreliable — use search_web to find a better answer                                   │
│  - If web search also fails: respond with 'Sorry, I could not find a reliable answer.'                          │
│  Agent: Answer Synthesiser                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'task_started' (expected 
'crew_kickoff_started')

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: e887c5ff-44ba-46d1-97d3-cba8f5751d55                                                                       │
│  Final Output: There are several AI tools that exist for clinical documentation, including Lindy AI, Heidi AI,  │
│  ScribePT, Upheal, Sully AI, and Tandem. These tools use artificial intelligence to capture patient             │
│  conversations and clinician dictation, auto-generate medical notes, and manage the entire documentation        │
│  workflow. Some of these tools, such as Lindy AI, can adapt to any workflow, support customizable templates,    │
│  and integrate with major EHRs. Others, such as Clinical Notes AI, can generate accurate and individualized     │
│  documentation in seconds and align with programs and standards of care. According to the website of Lindy AI,  │
│  it is the best AI medical documentation tool because it doesn't just take notes, but manages the entire        │
│  documentation workflow from start to finish. The information comes from various websites, including Lindy AI,  │
│  Heidi Health, and UChicago Medicine, which were retrieved using the search_web function. These websites        │
│  provide specific details and numbers, such as the fact that over 550 clinicians are using ambient clinical     │
│  documentation powered by Abridge AI, and that 90% of participating clinicians reported being able to give      │
│  undivided attention to patients after introducing the tool. Additionally, the websites mention that some of    │
│  these tools support dozens of medical specialties, from family medicine and pediatrics to orthopedics and      │
│  cardiology. Overall, the information retrieved from the web provides a detailed overview of the various AI     │
│  tools available for clinical documentation and their features.                                                 │
│  Source: web search                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

There are several AI tools that exist for clinical documentation, including Lindy AI, Heidi AI, ScribePT, Upheal, Sully AI, and Tandem. These tools use artificial intelligence to capture patient conversations and clinician dictation, auto-generate medical notes, and manage the entire documentation workflow. Some of these tools, such as Lindy AI, can adapt to any workflow, support customizable templates, and integrate with major EHRs. Others, such as Clinical Notes AI, can generate accurate and individualized documentation in seconds and align with programs and standards of care. According to the website of Lindy AI, it is the best AI medical documentation tool because it doesn't just take notes, but manages the entire documentation workflow from start to finish. The information comes from various websites, including Lindy AI, Heidi Health, and UChicago Medicine, which were retrieved using the search_web function. These websites provide specific details and numbers, such as the fact tha

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [26]:
trend_agent = Agent(
    role="Trend Analyst",
    goal="Summarise the latest news and trends related to the topic of the uploaded document",
    backstory=(
        "You are an expert analyst who searches the web for the latest "
        "developments related to any given topic. You distill the most important "
        "recent trends into a clear, concise summary."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
    tools=[search_web],
)

In [27]:
trend_task = Task(
    description=(
        "Search the web and summarise the 3 most important recent trends "
        "related to the topic of the uploaded document."
    ),
    expected_output="A clear bullet-point summary of 3 recent trends with sources.",
    agent=trend_agent,
)

In [28]:
trend_crew = Crew(
    agents=[trend_agent],
    tasks=[trend_task],
    verbose=True,
)

In [29]:
trend_result = trend_crew.kickoff()
print(trend_result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: a203ac6a-ee61-4b16-abcd-d52cb0ce7483                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search the web and summarise the 3 most important recent trends related to the topic of the uploaded     │
│  document.                                                                                                      │
│  ID: b12f2ebf-65c1-416a-819f-6aad791ff5e3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Trend Analyst                                                                                           │
│                                                                                                                 │
│  Task: Search the web and summarise the 3 most important recent trends related to the topic of the uploaded     │
│  document.                                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_web                                                                                               │
│  Args: {'question': 'latest news and trends related to the uploaded document'}                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_web executed with result: {'query': 'latest news and trends related to the uploaded document', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.adlibsoftware.com/news/the-big-8-trends...

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Search the web and summarise the 3 most important recent trends related to the topic of the uploaded     │
│  document.                                                                                                      │
│  Agent: Trend Analyst                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'task_started' (expected 
'crew_kickoff_started')

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: a203ac6a-ee61-4b16-abcd-d52cb0ce7483                                                                       │
│  Final Output: * The use of Artificial Intelligence (AI) and machine learning in document management is a       │
│  growing trend, with many companies incorporating AI-powered tools to automate tasks such as sorting, tagging,  │
│  and filing. (Source: https://docsvault.com/blog/document-management-trends/)                                   │
│  * Cloud-based document management systems are becoming increasingly popular, allowing for greater              │
│  collaboration and accessibility among team members. (Source:                                                   │
│  https://www.invensis.net/blog/document-management-trends)                                                      │
│  * The implementation of digital threads is another trend in document management, which involves creating a     │
│  digital record of all documents and communications related to a particular project or process. (Source:        │
│  https://www.adlibsoftware.com/news/the-big-8-trends-in-document-management-in-2025)                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

* The use of Artificial Intelligence (AI) and machine learning in document management is a growing trend, with many companies incorporating AI-powered tools to automate tasks such as sorting, tagging, and filing. (Source: https://docsvault.com/blog/document-management-trends/)
* Cloud-based document management systems are becoming increasingly popular, allowing for greater collaboration and accessibility among team members. (Source: https://www.invensis.net/blog/document-management-trends)
* The implementation of digital threads is another trend in document management, which involves creating a digital record of all documents and communications related to a particular project or process. (Source: https://www.adlibsoftware.com/news/the-big-8-trends-in-document-management-in-2025)


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [30]:
report_agent = Agent(
    role="Report Generator",
    goal="Collect insights from the uploaded document and generate a structured report",
    backstory=(
        "You are an expert analyst who reads uploaded documents "
        "and produces clear, structured reports summarising the key findings, "
        "challenges, and recommendations."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
    tools=[search_document],
)

In [31]:
report_task = Task(
    description=(
        "Using the search_document tool, search the uploaded document for information on: "
        "1. The main topic and purpose of the document "
        "2. Key findings or results presented "
        "3. Challenges or limitations discussed "
        "Then produce a structured markdown report with sections for each topic."
    ),
    expected_output=(
        "A structured markdown report with three sections: "
        "1. Document Overview "
        "2. Key Findings "
        "3. Challenges and Limitations. "
        "Each section should include key findings and statistics from the document."
    ),
    agent=report_agent,
)

In [32]:
report_crew = Crew(
    agents=[report_agent],
    tasks=[report_task],
    verbose=True,
)

In [33]:
report_result = report_crew.kickoff()
print(report_result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: da88ba4a-ac7f-4c75-a214-bd89c77250e0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Using the search_document tool, search the uploaded document for information on: 1. The main topic and   │
│  purpose of the document 2. Key findings or results presented 3. Challenges or limitations discussed Then       │
│  produce a structured markdown report with sections for each topic.                                             │
│  ID: 36ee1ba1-fe68-450d-9bdf-d9f2f03a2967                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Generator                                                                                        │
│                                                                                                                 │
│  Task: Using the search_document tool, search the uploaded document for information on: 1. The main topic and   │
│  purpose of the document 2. Key findings or results presented 3. Challenges or limitations discussed Then       │
│  produce a structured markdown report with sections for each topic.                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#3) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_document                                                                                          │
│  Args: {'question': 'main topic and purpose of the document, key findings or results presented, challenges or   │
│  limitations discussed'}                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_document executed with result: summarization tasks, and its transformative impact on the clinician’s potential, including but not
limited to reductions of administrative burden and improved patient-centered care.
1 Introduction
Pat...

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the search_document tool, search the uploaded document for information on: 1. The main topic and   │
│  purpose of the document 2. Key findings or results presented 3. Challenges or limitations discussed Then       │
│  produce a structured markdown report with sections for each topic.                                             │
│  Agent: Report Generator                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'task_started' (expected 
'crew_kickoff_started')

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: da88ba4a-ac7f-4c75-a214-bd89c77250e0                                                                       │
│  Final Output: # Document Overview                                                                              │
│  The main topic of the document is the use of Natural Language Processing (NLP) summarization models for        │
│  radiological reports. The purpose of the document is to discuss the potential of NLP summarization models in   │
│  reducing administrative burden and improving patient-centered care.                                            │
│                                                                                                                 │
│  # Key Findings                                                                                                 │
│  The document presents a case study that utilized NLP summarization models from various sources for             │
│  summarizing neuro-radiological reports. The study found that NLP summarization models can effectively          │
│  summarize lengthy and complex radiological reports, extracting necessary information and omitting unnecessary  │
│  details. The document also references a study by Siems et al. (2020) that assessed a structured chart review   │
│  methodology, which may be relevant to the use of NLP summarization models in clinical settings.                │
│                                                                                                                 │
│  # Challenges and Limitations                                                                                   │
│  The document discusses the challenges of unpacking necessary information from radiological reports, which can  │
│  be lengthy and complex. It also highlights the need for effective methods to summarize and extract relevant    │
│  information from these reports, which can be time-consuming and labor-intensive. However, the document does    │
│  not provide detailed information on the limitations of NLP summarization models or the challenges of           │
│  implementing these models in clinical practice. Further research is needed to fully understand the potential   │
│  benefits and limitations of using NLP summarization models for radiological reports.                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# Document Overview
The main topic of the document is the use of Natural Language Processing (NLP) summarization models for radiological reports. The purpose of the document is to discuss the potential of NLP summarization models in reducing administrative burden and improving patient-centered care.

# Key Findings
The document presents a case study that utilized NLP summarization models from various sources for summarizing neuro-radiological reports. The study found that NLP summarization models can effectively summarize lengthy and complex radiological reports, extracting necessary information and omitting unnecessary details. The document also references a study by Siems et al. (2020) that assessed a structured chart review methodology, which may be relevant to the use of NLP summarization models in clinical settings.

# Challenges and Limitations
The document discusses the challenges of unpacking necessary information from radiological reports, which can be lengthy and complex. It 

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯